* ¿Hay nulos en client_id, transaction_date, amount? ¿Cuántos y por qué (¿son transacciones inválidas o clientes nuevos sin historial completo?)
* ¿Hay duplicados exactos de transacción?
* ¿El rango de fechas coincide con los 4.5 años esperados? ¿Hay huecos sospechosos (ej: un mes entero sin transacciones — señal de error de carga, no de negocio)?
* ¿Hay montos negativos o cero? (podrían ser devoluciones — ¿tu diseño las contempla o las descarta?)

In [1]:
from pathlib import Path
import json

import pandas as pd

In [2]:
PROJECT_DIR = Path("..").resolve()

DATA_RAW_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_EDA_DIR = PROJECT_DIR / "outputs" / "m2" / "eda"

OUTPUT_EDA_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Carga datos
clients = pd.read_csv(DATA_RAW_DIR / "clients_raw.csv")
transactions = pd.read_csv(DATA_RAW_DIR / "transactions_raw.csv")
ground_truth = pd.read_csv(DATA_RAW_DIR / "_ground_truth.csv")

print(f"Clients shape: {clients.shape}")
print(f"Transactions shape: {transactions.shape}")
print(f"Ground Truth shape: {ground_truth.shape}")

Clients shape: (500, 7)
Transactions shape: (68483, 6)
Ground Truth shape: (500, 6)


Tenemos 500 clientes registrados.
Tenemos 68,483 transacciones históricas.
Tenemos un archivo auxiliar ground truth con 500 filas.

In [5]:
clients.head()

,client_id,business_name,store_type,zone,latitude,longitude,registration_date
0,CLI-0001,Pollería El Centro,restaurante,Chilca,-12.082178,-75.200853,2021-10-14
1,CLI-0002,Tienda Wanka,bodega,Chilca,-12.082362,-75.192667,2022-02-19
2,CLI-0003,Tienda Santa Rosa,bodega,San Carlos,-12.053894,-75.227877,2022-04-13
3,CLI-0004,Licores San Martín,licoreria,Chilca,-12.085464,-75.196285,2021-12-11
4,CLI-0005,Tienda El Progreso,bodega,El Tambo,-12.043708,-75.208575,2021-03-07


In [6]:
transactions.head()

,transaction_id,client_id,date,amount,sku_count,categories_purchased
0,TXN-000001,CLI-0177,2021-01-01,311.59,7,gaseosas|agua|jugos
1,TXN-000002,CLI-0484,2021-01-01,221.87,8,gaseosas|agua|jugos|energeticas
2,TXN-000003,CLI-0430,2021-01-01,69.21,4,gaseosas|agua|jugos
3,TXN-000004,CLI-0195,2021-01-01,106.35,5,gaseosas|agua
4,TXN-000005,CLI-0324,2021-01-02,152.73,5,gaseosas|agua|jugos


date                  → para Length, Recency y Frequency
amount                → para Monetary
categories_purchased  → para Volume

In [27]:
print(clients.columns)
print(transactions.columns)
print(ground_truth.columns)

Index(['client_id', 'business_name', 'store_type', 'zone', 'latitude',
       'longitude', 'registration_date'],
      dtype='str')
Index(['transaction_id', 'client_id', 'date', 'amount', 'sku_count',
       'categories_purchased'],
      dtype='str')
Index(['client_id', 'archetype_initial', 'trajectory_type', 'inflection_month',
       'expected_cluster_tentative', 'expected_cluster_jun2025'],
      dtype='str')


In [7]:
# nulos
print("Nulos en clients:" , clients.isnull().sum())
print("Nulos en transactions:" , transactions.isnull().sum())
print("Nulos en ground_truth:" , ground_truth.isnull().sum())

Nulos en clients: client_id            0
business_name        0
store_type           0
zone                 0
latitude             0
longitude            0
registration_date    0
dtype: int64
Nulos en transactions: transaction_id          0
client_id               0
date                    0
amount                  0
sku_count               0
categories_purchased    0
dtype: int64
Nulos en ground_truth: client_id                       0
archetype_initial               0
trajectory_type                 0
inflection_month              189
expected_cluster_tentative      0
expected_cluster_jun2025      500
dtype: int64


In [29]:
# revisar duplicados
print("Duplicados en clients:" , clients.duplicated().sum())
print("Duplicados en transactions:" , transactions.duplicated().sum())


Duplicados en clients: 0
Duplicados en transactions: 0


In [8]:
# convertir fechas
clients['registration_date'] = pd.to_datetime(clients['registration_date'],errors='coerce')
transactions['date'] = pd.to_datetime(transactions['date'],errors='coerce') # coerce para convertir fechas inválidas a NaT
registration_date_invalidos = clients['registration_date'].isna().sum()
date_invalidos = transactions['date'].isna().sum()
print(f"Fechas inválidas en clients: {registration_date_invalidos}")
print(f"Fechas inválidas en transactions: {date_invalidos}")

# revisar rango de las fechas 

print('primera fecha de transacción:', transactions['date'].min())
print('última fecha de transacción:', transactions['date'].max())

Fechas inválidas en clients: 0
Fechas inválidas en transactions: 0
primera fecha de transacción: 2021-01-01 00:00:00
última fecha de transacción: 2025-06-30 00:00:00


In [32]:
# categorias unicas
unique_categories = set(transactions['categories_purchased'].dropna().str.split('|').explode())
sorted_unique_categories = sorted(unique_categories)
print(f"Categorías únicas en transactions: {sorted_unique_categories}")

Categorías únicas en transactions: ['agua', 'energeticas', 'gaseosas', 'jugos']


In [9]:
clients_con_transactions = set(transactions["client_id"].unique())
clients_master = set(clients["client_id"].unique())

clientes_sin_transactiones = sorted(
    clients_master - clients_con_transactions
)

clientes_sin_transactiones

['CLI-0313', 'CLI-0315']

* Hay 2 clientes registrados que nunca compraron.
* Como LRFMV mide comportamiento de compra, un cliente sin compras no tiene evidencia suficiente para calcular Recency, Frequency, Monetary, Volume ni Length transaccional.

In [10]:
# resumen EDA Bronze
client_duplicates = clients["client_id"].duplicated().sum()
transaction_duplicates = transactions["transaction_id"].duplicated().sum()
invalid_registration_dates = clients["registration_date"].isna().sum()
invalid_transaction_dates = transactions["date"].isna().sum()
amount_le_zero = (transactions["amount"] <= 0).sum()
sku_count_le_zero = (transactions["sku_count"] <= 0).sum()

clients_with_transactions = set(transactions["client_id"].unique()) & set(clients["client_id"].unique())
clients_without_transactions = set(clients["client_id"].unique()) - set(transactions["client_id"].unique())
transactions_without_master_client = set(transactions["client_id"].unique()) - set(clients["client_id"].unique())

eda_summary = {
    "n_clients": len(clients),
    "n_transactiones": len(transactions),
    "n_ground_truth": len(ground_truth),
    "client_id_duplicados": int(client_duplicates),
    "transactiones_id_duplicados": int(transaction_duplicates),
    "invalid_registration_dates": int(invalid_registration_dates),
    "invalid_transaction_dates": int(invalid_transaction_dates),
    "amount_le_zero": int(amount_le_zero),
    "sku_count_le_zero": int(sku_count_le_zero),
    "clients_con_transactions": len(clients_with_transactions),
    "clients_sin_transactions": len(clients_without_transactions),
    "transactions_without_master_client": len(transactions_without_master_client),
    "transaction_date_min": str(transactions["date"].min().date()),
    "transaction_date_max": str(transactions["date"].max().date()),
    
}

eda_summary_path = OUTPUT_EDA_DIR / "eda_summary.json"
eda_summary_path.write_text(
    json.dumps(eda_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Resumen EDA guardado en: {eda_summary_path}")
eda_summary


Resumen EDA guardado en: ..\outputs\m2\eda\eda_summary.json


{'n_clients': 500,
 'n_transactiones': 68483,
 'n_ground_truth': 500,
 'client_id_duplicados': 0,
 'transactiones_id_duplicados': 0,
 'invalid_registration_dates': 0,
 'invalid_transaction_dates': 0,
 'amount_le_zero': 0,
 'sku_count_le_zero': 0,
 'clients_con_transactions': 498,
 'clients_sin_transactions': 2,
 'transactions_without_master_client': 0,
 'transaction_date_min': '2021-01-01',
 'transaction_date_max': '2025-06-30'}